In [1]:
!pip install langchain
!pip install openai
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.9 MB/s eta 0:00:00


In [2]:
import os
import openai

# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
import os
import openai

from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('kosa')

In [4]:
# 모델의 응답 >> 문자열로 parsing 하는데 사용
# 언어모델(lm) 출력(응답) >> 텍스트 형식으로 변환

from langchain_core.output_parsers import StrOutputParser

# 대화형 프롬프트 템플릿을 정의
from langchain_core.prompts import ChatPromptTemplate

# open ai의 언어모델 사용, 대화형 응답을 생성
from langchain_openai import ChatOpenAI

# 객체 생성
chat_prompt_template = ChatPromptTemplate.from_template('Tell me a short joke about {topic}')
chat_model = ChatOpenAI()
output_parser = StrOutputParser()
# message 의 content 추출 >> string (문자)변환

# 체인 정의 (아직 호출하지 않음)
chain = chat_prompt_template | chat_model | output_parser

In [5]:
chain.invoke({'topic': 'ice cream'})
# invoke 역할
# >> 인자를 dict로 넘겨줌 (dict {k:v} >> {"사과":"과일"})
# 이유:  prompt_template 가 dict를 받기 때문

'Why did the ice cream truck break down?\n\nBecause it had too many "scoops" of ice cream!'

In [6]:
# 객체를 생성하지 않는 경우
# >> 체인 정의할 때 바로 생성자 이용

chain = chat_prompt_template | ChatOpenAI() | StrOutputParser()
chain.invoke({'topic': 'ice cream'})


'Why did the ice cream truck break down?\n\nBecause it had too many "scoops"!'

In [9]:
# dict 말고 'ice cream'만 넣고 싶을 때

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

# runnables : 테스트 및 디버깅 용도
# RunnablePassthrough : 입력값을 그대로 출력값에 전달
# RunnableParallel : 여러 개의  Runnable 객체를 병렬로 실행
# RunnableLambda : 람다 함수나 임의의 함수 Runnable 감싸서 사용


chat_prompt_template = ChatPromptTemplate.from_template('Tell me a short joke about {topic}')
chat_model = ChatOpenAI()
output_parser = StrOutputParser()

chain  = {'topic': RunnablePassthrough()} | chat_prompt_template | chat_model | output_parser

In [10]:
chain.invoke("ice cream")

# "ice cream" > RunnablePassthrough() > {'topic': RunnablePassthrough()} > prompt_template

'Why did the ice cream truck break down? \n\nIt had too many "scoops"!'

In [11]:
chain  = RunnablePassthrough() | chat_prompt_template | chat_model | output_parser

In [12]:
chain.invoke("ice cream")

'Why did the ice cream truck break down? Because it had a meltdown!'

In [13]:
chain.invoke({'topic' :"ice cream"})

'Why did the ice cream truck break down?\n\nIt had too many sundaes to fundae!'

In [14]:
class Runnable:

    def __init__(self, func):
        self.func = func

    # 이 메서드는 파이썬의 비트 OR 연산자(|)를 오버로드합니다.
    def __or__(self, other):

        ## 다른 함수가 이 함수의 결과를 활용
        def chained_func(*args, **kwargs):
            return other(self.func(*args, **kwargs))

        #러너블 객체로 감싸게 됨
        return Runnable(chained_func)

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)

In [15]:
def add_five(x):
  return x + 5

def multipy_by_two(x):
  return x * 2

In [16]:
add_five(5)

10

In [17]:
multipy_by_two(5)

10

In [18]:
# Runnable 로 함수 감싸기
add_five = Runnable(add_five)
multiply_by_two = Runnable(multipy_by_two)

In [19]:
chain = add_five | multiply_by_two

chain(3)
# (3+5) * 2

16

In [20]:
chain = add_five.__or__(multiply_by_two)  # or >> |
chain(3)

16

In [21]:
def num2ko(x):
  return str(x) + '입니다.'

In [22]:
chain = add_five | multiply_by_two | num2ko

In [23]:
chain(3)

'16입니다.'

In [24]:
# 템플릿 만드는 방법

# 미션 : 어떤 주제에 대해서 사용자 레벨에 맞게 선택된 언어로 대화 예시를 만드는 앱을 개발!
# 미션 : [어떤] 주제에 대해서 사용자 [레벨]에 맞게 [선택된 언어]로 대화 예시를 만드는 앱을 개발!

# 기획
# "Please generate dialogue sentences in English on the topic of health for a beginner level."
# "Please generate dialogue sentences in Korean on the topic of AI for an intermediate level."

# 양식과 변수를 분리하기
# "Please generate three sentences in a dialogue in (English) on the topic of (health) for a (beginner) level."
# "Please generate three sentences in a dialogue in (Korean) on the topic of (AI) for an (intermediate) level."

# "Please generate three sentences in a dialogue in {language} on the topic of {topic} for a/an {level} level."

# 양식 조정하기
# "Please generate three sentences in a dialogue in {language} on the topic of {topic} for a level of {level}."

In [25]:
from langchain_core.runnables import RunnablePassthrough

In [27]:
template =\
"Please generate three sentences in a dialogue in {language} on the topic of {topic} for a level of {level}"

In [28]:
# cf) chatgpt
# [초보자 수준]으로 [저녁메뉴] 관련된 3개의 문장으로 구성된 대화지문 [한국어]로 생성해줘.

In [29]:
chat_prompt_template = ChatPromptTemplate.from_template(template)

In [30]:
chain = chat_prompt_template | chat_model | StrOutputParser()

In [31]:
# 변수는 dict 로 넣어야 함
output = chain.invoke( {"language" : "English", "topic": "dinner menu", "level":"beginner"} )
print(output)

Person 1: "What's for dinner tonight?"
Person 2: "We have chicken, salad, and rice on the menu."
Person 1: "Sounds delicious! I can't wait to eat."


In [32]:
# 변수 중에 시스템에 입력해야 할 것과 사용자가 입력해야 할 것 분리
# language : 설정
# topic : 바뀐다
# level: 설정

def get_learning_language(_):
  print("###")
  print(_)
  print("in get_learning_language")
  print("###")
  return "English"


def get_learning_level(_):
  print("###")
  print(_)
  print("in get_learning_level")
  print("###")
  return "beginner"

# '_' : 함수 내에서 사용되지 않는 값을 받을 때
# 즉 여기서는 입력값을 사용하지 않고 항상 "beginner"를 반환

In [33]:
# 호출 방법 1

template =\
"Please generate three sentences in a dialogue in {language} on the topic of {topic} for a level of {level}"

prompt_template = ChatPromptTemplate.from_template(template)

chain = prompt_template | chat_model | StrOutputParser()

# invoke 하기 전에 필요한 정보를 dict 로 구성하는 방법
output =\
chain.invoke({'language': get_learning_language(''), 'topic': "travel", "level": get_learning_level('') })

print(output)

###

in get_learning_language
###
###

in get_learning_level
###
1. Person A: "Do you like to travel?"
   Person B: "Yes, I love going to new places and exploring different cultures."

2. Person A: "What's your favorite travel destination?"
   Person B: "I really enjoy visiting beach resorts and relaxing by the ocean."

3. Person A: "Have you ever been on a road trip?"
   Person B: "No, but I would love to go on a road trip and see all the different sights along the way."


In [34]:
# 방법 2 Runnable 사용

from langchain_core.runnables import RunnablePassthrough

template =\
"Please generate three sentences in a dialogue in {language} on the topic of {topic} for a level of {level}"

chat_prompt_template = ChatPromptTemplate.from_template(template)

# 함수의 인자는 '무조건 1개' 이어야 함

chain = (
  RunnablePassthrough.assign(language=get_learning_language,
                           level=get_learning_level
                           )
  | chat_prompt_template
  | chat_model
  | StrOutputParser()
)

# {'topic': 'travel'} >> dict language, level 추가 >> {"topic",..."language", ...."level"...} dict 완성

output = chain.invoke({'topic': "travel"})   # 변수 1개

print(output)


###
{'topic': 'travel'}
in get_learning_language
###
###
{'topic': 'travel'}
in get_learning_level
###
Person A: "Have you ever traveled to a different country?"
Person B: "Yes, I went to Italy last summer. It was amazing."
Person A: "I want to travel too, but I don't know where to go. Any suggestions?"


In [35]:
chain

RunnableAssign(mapper={
  language: RunnableLambda(get_learning_language),
  level: RunnableLambda(get_learning_level)
})
| ChatPromptTemplate(input_variables=['language', 'level', 'topic'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language', 'level', 'topic'], input_types={}, partial_variables={}, template='Please generate three sentences in a dialogue in {language} on the topic of {topic} for a level of {level}'), additional_kwargs={})])
| ChatOpenAI(profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': 

In [36]:
chain = (
  RunnablePassthrough.assign(language=get_learning_language,
                           level=get_learning_level
                           )
  | chat_prompt_template
  | chat_model
  | StrOutputParser()
)

# {'topic': 'travel'} >> dict language, level 추가 >> {"topic",..."language", ...."level"...} dict 완성

output = chain.invoke({'topic': "weather"})   # 변수 1개

print(output)

######
{'topic': 'weather'}
in get_learning_level
###

{'topic': 'weather'}
in get_learning_language
###
A: How's the weather today?
B: It's sunny and warm. I love it!
A: Me too! Let's go for a walk outside.


In [37]:
chain

RunnableAssign(mapper={
  language: RunnableLambda(get_learning_language),
  level: RunnableLambda(get_learning_level)
})
| ChatPromptTemplate(input_variables=['language', 'level', 'topic'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language', 'level', 'topic'], input_types={}, partial_variables={}, template='Please generate three sentences in a dialogue in {language} on the topic of {topic} for a level of {level}'), additional_kwargs={})])
| ChatOpenAI(profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': 